# Universal Knowledge Distillation
Notebook ibrido che supporta sia il task di **Summarization** (SAMSum) che di **Question Answering / Instruction Following** (Dolly-15k).
Segue fedelmente l'approccio di `Summarization.ipynb`: pipeline HuggingFace per la generazione offline del Teacher, `SFTTrainer` per il training dello Student, e valutazione su ROUGE, BERTScore e Perplexity — con l'aggiunta di metriche di performance (Latenza, Parametri).

<a target="_blank" href="https://colab.research.google.com/github/WholeNow/KnowledgeDistillator/blob/main/Universal_KD.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

## Step 0 — Setup e Installazioni

In [ ]:
!pip install -q transformers datasets trl evaluate rouge_score bert_score matplotlib accelerate bitsandbytes

## Step 0b — Importazione Librerie

In [ ]:
# All imports
import os
import torch
import math
import random
import json
from datasets import load_dataset, Dataset, load_from_disk
from transformers import pipeline, AutoTokenizer, GenerationConfig, AutoModelForCausalLM
from transformers.pipelines.pt_utils import KeyDataset
from trl import SFTTrainer, SFTConfig
from tqdm.auto import tqdm
import evaluate

## Configurazione
**Imposta qui tutti i parametri dell'esperimento prima di eseguire le celle successive.**

In [ ]:
# ============================================================
#                     CONFIGURAZIONE
# ============================================================

# TASK: 'summarization' → SAMSum | 'qa' → Dolly-15k
TASK_TYPE = "summarization"

# TEACHER: scegli uno dei due
# "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  → più veloce, ottimo per summarization
# "Qwen/Qwen2.5-1.5B-Instruct"          → più preciso su QA
TEACHER_MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# STUDENT: fisso per questo esperimento
STUDENT_MODEL_ID = "HuggingFaceTB/SmolLM-135M"

# Numero di sample da usare per il training del Teacher (generazione etichette)
# Usa 'all' per l'intero dataset, oppure un intero es. 500
MAX_TRAIN_SAMPLES = 'all'

# Numero di sample per la valutazione finale
# Usa 'all' oppure un intero es. 100
MAX_TEST_SAMPLES = 'all'

# Batch size per la pipeline del Teacher (generazione etichette offline)
TEACHER_PIPELINE_BATCH_SIZE = 16

# Iperparametri training Student
STUDENT_BATCH_SIZE = 4
GRAD_ACCUMULATION = 4
EPOCHS = 3
LEARNING_RATE = 1e-5
MAX_SEQ_LENGTH = 1024

# Cartelle di output (generate automaticamente o precaricate dall'utente)
TEACHER_DATASET_DIR = f"./dataset_distilled_{TASK_TYPE}"
STUDENT_OUTPUT_DIR  = f"./student_distilled_{TASK_TYPE}"
STUDENT_FINAL_DIR   = f"./student_distilled_{TASK_TYPE}_final"


## Step 1 — Generazione Dataset Distillato (Teacher)
Usiamo la `pipeline` di HuggingFace con batching nativo per generare le pseudo-label del Teacher in modo efficiente e salvarle su disco.
Se il file è già presente, viene caricato direttamente.

In [ ]:
# ── Cleanup VRAM/RAM per evitare OOM su riavvio/interruzione ──
import gc
import torch
for var in ['generator', 'student_model', 'model', 'trainer']:
    if var in globals():
        del globals()[var]
gc.collect()
torch.cuda.empty_cache()

# ── Configurazione prompt in base al task ──────────────────
if TASK_TYPE == "summarization":
    raw_dataset = load_dataset("knkarthick/samsum", split="train")
    
    def build_prompt(example):
        messages = [
            {"role": "system", "content": "You are a highly accurate summarization assistant. Provide a concise summary of the following conversation."},
            {"role": "user",   "content": f"Summarize this dialogue:\n\n{example['dialogue']}"}
        ]
        return {"prompt": tokenizer_teacher.apply_chat_template(messages, tokenize=False, add_generation_prompt=True),
                "target": example["summary"],
                "input_text": example["dialogue"]}

elif TASK_TYPE == "qa":
    _full = load_dataset("databricks/databricks-dolly-15k", split="train")
    _split = _full.train_test_split(test_size=0.1, seed=42)
    raw_dataset = _split["train"]
    
    def build_prompt(example):
        context = example.get("context", "") or ""
        # Limita la lunghezza del contesto per evitare prompt eccessivamente lunghi e OOM
        if len(context) > 3000:
            context = context[:3000] + "... [TRUNCATED]"
        instruction = example["instruction"]
        user_content = (f"Context: {context}\n\nInstruction: {instruction}" if context.strip()
                        else f"Instruction: {instruction}")
        input_text = (f"Context: {context[:300]}...\nInstruction: {instruction}" if context.strip()
                      else f"Instruction: {instruction}")
        messages = [
            {"role": "system", "content": "You are a helpful and concise AI assistant. Follow the user's instruction."},
            {"role": "user",   "content": user_content}
        ]
        return {"prompt": tokenizer_teacher.apply_chat_template(messages, tokenize=False, add_generation_prompt=True),
                "target": example["response"],
                "input_text": input_text}
else:
    raise ValueError("TASK_TYPE non valido. Scegli 'summarization' o 'qa'.")

# ── Limitazione sample ─────────────────────────────────────
if MAX_TRAIN_SAMPLES != 'all':
    raw_dataset = raw_dataset.select(range(min(int(MAX_TRAIN_SAMPLES), len(raw_dataset))))

print(f"Task: {TASK_TYPE} | Teacher: {TEACHER_MODEL_ID} | Train samples: {len(raw_dataset)}")

# ── Inizializzazione tokenizer Teacher ────────────────────
tokenizer_teacher = AutoTokenizer.from_pretrained(TEACHER_MODEL_ID)
if tokenizer_teacher.pad_token is None:
    tokenizer_teacher.pad_token = tokenizer_teacher.eos_token
tokenizer_teacher.truncation_side = "left"

# ── Costruzione dataset prompt ────────────────────────────
prepared = raw_dataset.map(build_prompt)
prompt_dataset = Dataset.from_dict({"prompt": prepared["prompt"]})

# ── Generazione etichette Teacher (con cache su disco) ────
if os.path.exists(TEACHER_DATASET_DIR):
    print("Dataset distillato già presente su disco. Caricamento...")
    distilled_dataset = load_from_disk(TEACHER_DATASET_DIR)
else:
    print("Avvio generazione pseudo-label con pipeline Teacher...")

    # Carica l'intero modello Teacher via pipeline (gestisce batching e memoria internamente)
    partial_file = f"{TEACHER_DATASET_DIR}_partial.jsonl"
    
    # Resume: leggi risposte già generate
    teacher_summaries = []
    if os.path.exists(partial_file):
        print("Trovato file parziale, ripresa in corso...")
        with open(partial_file, "r") as fp:
            for line in fp:
                teacher_summaries.append(json.loads(line)["teacher_summary"])
        print(f"Già elaborati: {len(teacher_summaries)} / {len(prepared)}")

    start_idx = len(teacher_summaries)

    if start_idx < len(prepared):
        generator = pipeline(
            "text-generation",
            model=TEACHER_MODEL_ID,
            dtype=torch.float16,
            device_map="auto",
            batch_size=TEACHER_PIPELINE_BATCH_SIZE
        )

        gen_config = GenerationConfig(
            max_new_tokens=128,
            do_sample=False,
            return_full_text=False,
        )

        remaining_prompts = Dataset.from_dict({"prompt": prepared["prompt"][start_idx:]})

        with open(partial_file, "a") as fp:
            for out in tqdm(
                generator(KeyDataset(remaining_prompts, "prompt"), generation_config=gen_config, preprocess_params={"truncation": True, "max_length": 1920}),
                total=len(remaining_prompts),
                desc="Distillazione Teacher"
            ):
                text = out[0]["generated_text"].strip()
                teacher_summaries.append(text)
                fp.write(json.dumps({"teacher_summary": text}) + "\n")
                fp.flush()

        # Libera memoria GPU del Teacher prima di caricare lo Student
        del generator
        torch.cuda.empty_cache()

    # Assembla e salva il dataset finale
    distilled_dataset = prepared.add_column("teacher_summary", teacher_summaries)
    distilled_dataset.save_to_disk(TEACHER_DATASET_DIR)
    if os.path.exists(partial_file):
        os.remove(partial_file)
    print(f"Dataset salvato in {TEACHER_DATASET_DIR}")


## Step 2 — Training dello Student
Fine-tuning di `SmolLM-135M` sulle pseudo-label generate dal Teacher tramite `SFTTrainer`.

In [ ]:
# ── Cleanup VRAM/RAM per evitare OOM su riavvio/interruzione ──
import gc
import torch
for var in ['generator', 'student_model', 'model', 'trainer']:
    if var in globals():
        del globals()[var]
gc.collect()
torch.cuda.empty_cache()

# 1. Inizializzazione tokenizer Student
tokenizer_student = AutoTokenizer.from_pretrained(STUDENT_MODEL_ID)
if tokenizer_student.pad_token is None:
    tokenizer_student.pad_token = tokenizer_student.eos_token

# Iniezione forzata del ChatML template se assente (es. SmolLM-135M base)
if tokenizer_student.chat_template is None:
    tokenizer_student.chat_template = (
        "{% for message in messages %}"
        "<|im_start|>{{ message['role'] }}\n"
        "{{ message['content'] }}<|im_end|>\n"
        "{% endfor %}"
        "{% if add_generation_prompt %}"
        "<|im_start|>assistant\n"
        "{% endif %}"
    )

# 2. Caricamento modello Student in FP32 (più stabile)
student_model = AutoModelForCausalLM.from_pretrained(
    STUDENT_MODEL_ID,
    dtype=torch.float32,
    device_map="auto"
)

# 3. Formato conversazionale (messages) identico a Summarization.ipynb
def format_example(example, task_type="generic"):
    """
    Converte un record del dataset nel formato 'messages' compatibile con TRL/SFTTrainer.
    task_type può essere:
        - "summarization"
        - "qa"
    """

    # 1. SYSTEM MESSAGE
    if task_type == "summarization":
        system_msg = (
            "You are a highly accurate summarization assistant. "
            "Provide a concise summary of the following conversation."
        )
    else:
        system_msg = (
            "You are a helpful and concise AI assistant. "
            "Follow the user's instruction."
        )

    # 2. USER MESSAGE
    if task_type == "summarization":
        # Usa il campo 'dialogue' come nella tua seconda funzione
        user_msg = f"Summarize this dialogue:\n\n{example['dialogue']}"
    else:
        # Usa il campo 'input_text' come nella tua prima funzione
        user_msg = example["input_text"]

    # 3. ASSISTANT MESSAGE (teacher output)
    assistant_msg = example["teacher_summary"]

    # 4. Restituzione nel formato TRL
    return {
        "messages": [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": assistant_msg},
        ]
    }
# Se il dataset non esiste in memoria allora cerca se esiste su disco, altrimenti solleva un errore
if 'distilled_dataset' in globals():
    pc_dataset = distilled_dataset.map(lambda x: format_example(x, TASK_TYPE), remove_columns=distilled_dataset.column_names)
elif os.path.exists(TEACHER_DATASET_DIR):
    distilled_dataset = load_from_disk(TEACHER_DATASET_DIR)
    pc_dataset = distilled_dataset.map(lambda x: format_example(x, TASK_TYPE), remove_columns=distilled_dataset.column_names)
else:
    raise ValueError(f"Dataset {TEACHER_DATASET_DIR} non trovato. Assicurati che la generazione del Teacher sia completata correttamente.")

# 4. SFTConfig (identico a Summarization.ipynb, con parametri dalla cella di config)
training_args = SFTConfig(
    output_dir=STUDENT_OUTPUT_DIR,
    per_device_train_batch_size=STUDENT_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUMULATION,
    gradient_checkpointing=True,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=50,
    logging_steps=10,
    num_train_epochs=EPOCHS,
    fp16=True,           # Come Summarization.ipynb (stabile su T4/A100)
    optim="adamw_torch_fused",
    report_to="none",
    max_length=MAX_SEQ_LENGTH,
    completion_only_loss=False,  # Come Summarization.ipynb
)

trainer = SFTTrainer(
    model=student_model,
    args=training_args,
    train_dataset=pc_dataset,
    processing_class=tokenizer_student,
)

print("Avvio addestramento...")
trainer.train()

trainer.save_model(STUDENT_FINAL_DIR)
tokenizer_student.save_pretrained(STUDENT_FINAL_DIR)
print(f"Modello salvato in {STUDENT_FINAL_DIR}")


## Step 3 — Valutazione
Confrontiamo **Teacher**, **Student Baseline (zero-shot)** e **Student Distillato**.
Metriche: ROUGE-1, ROUGE-2, ROUGE-L, BERTScore-F1, Perplexity, Latenza/Token, Parametri.

In [ ]:
# ── Cleanup VRAM/RAM per evitare OOM su riavvio/interruzione ──
import gc
import torch
for var in ['generator', 'student_model', 'model', 'trainer']:
    if var in globals():
        del globals()[var]
gc.collect()
torch.cuda.empty_cache()

import time

device = "cuda" if torch.cuda.is_available() else "cpu"
rouge_metric = evaluate.load("rouge")
bert_metric  = evaluate.load("bertscore")

# ── Test set ──────────────────────────────────────────────
if TASK_TYPE == "summarization":
    test_raw = load_dataset("knkarthick/samsum", split="test")
    def get_prompt_and_target(sample):
        messages = [
            {"role": "system", "content": "You are a highly accurate summarization assistant. Provide a concise summary of the following conversation."},
            {"role": "user",   "content": f"Summarize this dialogue:\n\n{sample['dialogue']}"}
        ]
        return messages, sample["summary"], sample["dialogue"]
elif TASK_TYPE == "qa":
    _full = load_dataset("databricks/databricks-dolly-15k", split="train")
    _split = _full.train_test_split(test_size=0.1, seed=42)
    test_raw = _split["test"]
    def get_prompt_and_target(sample):
        context = sample.get("context", "") or ""
        uc = (f"Context: {context}\n\nInstruction: {sample['instruction']}" if context.strip()
              else f"Instruction: {sample['instruction']}")
        messages = [
            {"role": "system", "content": "You are a helpful and concise AI assistant. Follow the user's instruction."},
            {"role": "user",   "content": uc}
        ]
        return messages, sample["response"], uc

if MAX_TEST_SAMPLES != 'all':
    test_raw = test_raw.select(range(min(int(MAX_TEST_SAMPLES), len(test_raw))))

print(f"Test samples: {len(test_raw)}")

# ── Funzione di valutazione (stessa struttura di Summarization.ipynb) ──
def evaluate_model(model_path):
    print(f"\nValutazione: {model_path}")
    tok = AutoTokenizer.from_pretrained(model_path)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    if tok.chat_template is None:
        tok.chat_template = (
            "{% for message in messages %}"
            "<|im_start|>{{ message['role'] }}\n"
            "{{ message['content'] }}<|im_end|>\n"
            "{% endfor %}"
            "{% if add_generation_prompt %}"
            "<|im_start|>assistant\n"
            "{% endif %}"
        )

    model = AutoModelForCausalLM.from_pretrained(model_path, dtype=torch.float32).to(device)
    model.eval()

    predictions, references = [], []
    total_loss, total_time, total_gen_tokens = 0.0, 0.0, 0

    for sample in tqdm(test_raw, desc=f"Eval {model_path}"):
        messages, target, _ = get_prompt_and_target(sample)
        prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tok(prompt, return_tensors="pt", padding=True).to(device)

        # Perplexity: forward pass sul testo completo (prompt + risposta reale)
        full_text   = prompt + target + tok.eos_token
        full_inputs = tok(full_text, return_tensors="pt").to(device)
        with torch.no_grad():
            loss_out = model(**full_inputs, labels=full_inputs["input_ids"])
            total_loss += loss_out.loss.item()

            # Generazione per ROUGE/BERTScore
            t0 = time.time()
            outputs = model.generate(
                **inputs,
                max_new_tokens=128,
                do_sample=False,
                pad_token_id=tok.pad_token_id
            )
            total_time += time.time() - t0

        gen_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        total_gen_tokens += len(gen_tokens)
        gen_text = tok.decode(gen_tokens, skip_special_tokens=True).strip()
        predictions.append(gen_text)
        references.append(target)

    rouge_res = rouge_metric.compute(predictions=predictions, references=references)
    bert_res  = bert_metric.compute(predictions=predictions, references=references, lang="en")
    avg_bert_f1 = sum(bert_res["f1"]) / len(bert_res["f1"])

    avg_loss   = total_loss / len(test_raw)
    perplexity = math.exp(avg_loss) if avg_loss < 20 else float("inf")
    ms_per_tok = (total_time * 1000) / total_gen_tokens if total_gen_tokens > 0 else 0
    params_m   = sum(p.numel() for p in model.parameters()) / 1e6

    del model
    torch.cuda.empty_cache()

    return {
        "Perplexity":        perplexity,
        "ROUGE-1":           rouge_res["rouge1"],
        "ROUGE-2":           rouge_res["rouge2"],
        "ROUGE-L":           rouge_res["rougeL"],
        "BERTScore-F1":      avg_bert_f1,
        "Latency/Token (ms)": ms_per_tok,
        "Parameters (M)":    params_m,
        "predictions":       predictions,
    }

# ── Esecuzione (Teacher + Baseline + Distilled) ─────────
metrics_teacher   = evaluate_model(TEACHER_MODEL_ID)
metrics_baseline  = evaluate_model(STUDENT_MODEL_ID)
metrics_distilled = evaluate_model(STUDENT_FINAL_DIR)

print("\n=== RISULTATI COMPARATIVI ===")
for name, m in [("Teacher", metrics_teacher), ("Student Baseline", metrics_baseline), ("Student Distilled", metrics_distilled)]:
    print(f"\n{name}:")
    for k, v in m.items():
        if k != "predictions":
            print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")


## Step 3b — Grafici per Relazione

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plot_metrics = ["Perplexity", "ROUGE-L", "BERTScore-F1", "Latency/Token (ms)"]
models       = ["Teacher", "Student Baseline", "Student Distilled"]
all_results  = [metrics_teacher, metrics_baseline, metrics_distilled]
colors       = ["steelblue", "lightcoral", "mediumseagreen"]

fig, axes = plt.subplots(1, len(plot_metrics), figsize=(14, 5))
fig.suptitle(f"Knowledge Distillation — Task: {TASK_TYPE.upper()}", fontsize=14, fontweight="bold")

for ax, metric in zip(axes, plot_metrics):
    vals = [r[metric] for r in all_results]
    bars = ax.bar(models, vals, color=colors, edgecolor="white", linewidth=0.5)
    ax.set_title(metric, fontsize=11)
    ax.set_xticks(range(len(models)))
    ax.set_xticklabels([m.replace(" ", "\n") for m in models], fontsize=9)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(vals)*0.01,
                f"{val:.2f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig(f"kd_results_{TASK_TYPE}.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Grafico salvato come kd_results_{TASK_TYPE}.png")


## Step 3c — Ispezione Qualitativa
Stampa di 3 esempi casuali a confronto tra i tre modelli.

In [ ]:
# Caricamento modello distillato in fp16 per l'ispezione qualitativa (come Summarization.ipynb)
tok_qual   = AutoTokenizer.from_pretrained(STUDENT_FINAL_DIR)
model_qual = AutoModelForCausalLM.from_pretrained(STUDENT_FINAL_DIR, dtype=torch.float16, device_map="auto")

samples = random.sample(list(test_raw), min(3, len(test_raw)))

print("=== ISPEZIONE QUALITATIVA DEGLI OUTPUT ===\n")
for i, sample in enumerate(samples, 1):
    messages, target, input_text = get_prompt_and_target(sample)
    prompt = tok_qual.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok_qual(prompt, return_tensors="pt").to(model_qual.device)

    with torch.no_grad():
        outputs = model_qual.generate(
            **inputs, max_new_tokens=128, do_sample=False,
            pad_token_id=tok_qual.eos_token_id
        )
    gen_text = tok_qual.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)

    print(f"--- SAMPLE {i} ---")
    print(f"INPUT:\n{input_text.strip()}\n")
    print(f"TARGET IDEALE (Human):\n{target.strip()}\n")
    print(f"GENERAZIONE STUDENT DISTILLATO:\n{gen_text.strip()}\n")
    print("STUDENT BASELINE prediction:", metrics_baseline["predictions"][test_raw.to_list().index(sample)] if hasattr(test_raw, 'to_list') else "(esegui con indice)")
    print("=" * 60 + "\n")
